# ️ Digital Image Processing: Elementary Methods

**Addis Ababa University** | Computer Vision — Individual Assignment

**Author:** Tinsae Tadesse

**Submitted to:** Dr. Fantahun (PhD)


## Overview

This notebook implements **7 fundamental intensity transformation and histogram processing methods** with academic rigor, referencing foundational literature:

| # | Method | Formula | Reference |
|---|--------|---------|-----------|
| 1 | Image Negative | $s = (L-1) - r$ | Gonzalez & Woods §3.2.1 |
| 2 | Gamma Correction | $s = c \cdot r^\gamma$ | Poynton (1998); IEC 61966 |
| 3 | Logarithmic Transform | $s = c \cdot \log(1 + r)$ | Stockham (1972) |
| 4 | Contrast Stretching | Piecewise linear | Gonzalez & Woods §3.2.4 |
| 5 | Histogram Equalization | CDF-based mapping | Pizer et al. (1987) |
| 6 | Intensity Level Slicing | Range highlighting | Gonzalez & Woods §3.2.5 |
| 7 | Bit Plane Slicing | Binary decomposition | Gonzalez & Woods §3.2.6 |

All methods handle both **RGB** and **grayscale** images. 

> **Key References:**
> - Gonzalez, R.C. & Woods, R.E. (2018). *Digital Image Processing*, 4th Ed.
> - Stockham, T.G. Jr. (1972). *Image Processing in the Context of a Visual Model*. Proc. IEEE.
> - Pizer, S.M. et al. (1987). *Adaptive Histogram Equalization and Its Variations*. CVGIP.
> - Poynton, C. (1998). *The Rehabilitation of Gamma*. SPIE.

## 0. Setup & Dependencies

In [ ]:
# Install dependencies
!pip install -q numpy opencv-python-headless matplotlib scikit-image Pillow

In [ ]:
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from skimage import data, color, img_as_ubyte, img_as_float
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# --- Sleek dark plot style ---
plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor': '#16213e',
    'axes.edgecolor': '#e94560',
    'axes.labelcolor': '#eee',
    'text.color': '#eee',
    'xtick.color': '#aaa',
    'ytick.color': '#aaa',
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
})

print('All imports successful!')

## 0.1 Utility Functions

In [ ]:
# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def is_grayscale(img):
    """Check if an image is grayscale."""
    return len(img.shape) == 2 or (len(img.shape) == 3 and img.shape[2] == 1)

def to_grayscale(img):
    """Convert RGB to grayscale; return as-is if already grayscale."""
    if is_grayscale(img):
        return img.squeeze() if len(img.shape) == 3 else img
    return cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

def ensure_uint8(img):
    """Safely convert image to uint8."""
    if img.dtype == np.uint8:
        return img
    if img.dtype in [np.float32, np.float64]:
        if img.max() <= 1.0:
            return (img * 255).clip(0, 255).astype(np.uint8)
        return img.clip(0, 255).astype(np.uint8)
    if img.dtype == np.bool_:
        return (img.astype(np.uint8)) * 255
    return img.clip(0, 255).astype(np.uint8)

def _show_image(ax, img, title):
    """Display an image on a matplotlib axis."""
    if is_grayscale(img):
        ax.imshow(img.squeeze(), cmap='gray', vmin=0, vmax=255)
    else:
        ax.imshow(img)
    ax.set_title(title, fontsize=11, pad=8)
    ax.axis('off')

def _plot_histogram(ax, img, title):
    """Plot histogram of an image."""
    ax.set_title(title, fontsize=10, pad=6)
    if is_grayscale(img):
        hist = cv2.calcHist([img.squeeze()], [0], None, [256], [0, 256])
        ax.fill_between(range(256), hist.flatten(), alpha=0.7, color='#e94560')
        ax.plot(hist, color='#e94560', linewidth=0.8)
    else:
        for i, (ch, cn) in enumerate(zip(['#ff4444', '#44ff44', '#4488ff'], ['R', 'G', 'B'])):
            hist = cv2.calcHist([img], [i], None, [256], [0, 256])
            ax.plot(hist, color=ch, linewidth=1.0, alpha=0.8, label=cn)
        ax.legend(fontsize=8, loc='upper right', facecolor='#16213e', edgecolor='#444', labelcolor='#eee')
    ax.set_xlim([0, 255])
    ax.set_xlabel('Intensity', fontsize=9)
    ax.set_ylabel('Frequency', fontsize=9)
    ax.grid(True, alpha=0.15)

def show_comparison(original, processed, title,
                    original_label='Original', processed_label='Processed',
                    subtitle=None):
    """Side-by-side comparison of original and processed images."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(title, fontsize=16, fontweight='bold', color='#e94560', y=0.98)
    if subtitle:
        fig.text(0.5, 0.93, subtitle, ha='center', fontsize=9, color='#aaa', style='italic')
    _show_image(axes[0], original, original_label)
    _show_image(axes[1], processed, processed_label)
    plt.tight_layout(rect=[0, 0, 1, 0.90])
    plt.show()

def show_multi(images, labels, title, cols=4, subtitle=None):
    """Grid comparison of multiple images."""
    n = len(images)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
    fig.suptitle(title, fontsize=16, fontweight='bold', color='#e94560', y=0.98)
    if subtitle:
        fig.text(0.5, 0.94, subtitle, ha='center', fontsize=9, color='#aaa', style='italic')
    if rows == 1 and cols == 1:
        axes = np.array([axes])
    axes = np.atleast_2d(axes)
    for idx in range(rows * cols):
        r, c = idx // cols, idx % cols
        ax = axes[r, c]
        if idx < n:
            _show_image(ax, images[idx], labels[idx])
        else:
            ax.axis('off')
    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()

def show_with_histogram(original, processed, title,
                        original_label='Original', processed_label='Processed',
                        subtitle=None):
    """Side-by-side images with their histograms below."""
    fig = plt.figure(figsize=(14, 10))
    gs = GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)
    fig.suptitle(title, fontsize=16, fontweight='bold', color='#e94560', y=0.98)
    if subtitle:
        fig.text(0.5, 0.94, subtitle, ha='center', fontsize=9, color='#aaa', style='italic')
    ax1 = fig.add_subplot(gs[0, 0]); _show_image(ax1, original, original_label)
    ax2 = fig.add_subplot(gs[0, 1]); _show_image(ax2, processed, processed_label)
    ax3 = fig.add_subplot(gs[1, 0]); _plot_histogram(ax3, original, f'{original_label} Histogram')
    ax4 = fig.add_subplot(gs[1, 1]); _plot_histogram(ax4, processed, f'{processed_label} Histogram')
    plt.show()

print('Utility functions loaded!')

## 0.2 Load Test Images

I use classic images from the image processing literature.

In [ ]:
# Load classic test images from scikit-image
img_camera = data.camera()       # Grayscale 512x512 — classic benchmark
img_moon = data.moon()           # Grayscale — low contrast, perfect for HE
img_astronaut = data.astronaut() # RGB 512x512 — NASA astronaut
img_coffee = data.coffee()       # RGB — great for gamma demos

# Here we'll use the Chelsea cat as an additional RGB test image
img_chelsea = data.chelsea()     # RGB — cat image, rich detail

# Load from Colab runtime
import os
if os.path.exists('jebena.png'):
    img_jebena = cv2.cvtColor(cv2.imread('jebena.png'), cv2.COLOR_BGR2RGB)
else:
    img_jebena = img_coffee # Fallback if not found

# Display all test images
test_images = [img_camera, img_moon, img_astronaut, img_coffee, img_chelsea, img_jebena]
test_labels = [
    f'Cameraman\n{img_camera.shape}',
    f'Moon\n{img_moon.shape}',
    f'Astronaut\n{img_astronaut.shape}',
    f'Coffee\n{img_coffee.shape}',
    f'Chelsea\n{img_chelsea.shape}'
    ,f'Jebena\n{img_jebena.shape}'
]

show_multi(test_images, test_labels,
           'Test Images from Image Processing Literature',
           cols=6,
           subtitle='Classic benchmarks used across thousands of research papers')

# 1. ️ Image Negative

## Theory

The **negative transformation** reverses intensity levels:

$$s = (L - 1) - r$$

where $L = 256$ for 8-bit images. This is the digital equivalent of a **photographic film negative**.

### Properties
- **Self-inverse**: applying the negative twice returns the original
- **Per-channel** for RGB: $s_c = 255 - r_c$ for $c \in \{R, G, B\}$
- **O(M×N)** complexity — single pass over all pixels
- Can be implemented as a **256-entry LUT** for O(1) per pixel

### Applications
- **Medical imaging**: Enhancing white/gray detail in mammograms and X-rays
- **Astronomy**: Revealing faint structures in dark sky images

> *Reference: Gonzalez & Woods (2018), §3.2.1*

In [ ]:
# ============================================================
# 1. IMAGE NEGATIVE
# s = (L-1) - r, where L = 256 for 8-bit images
# ============================================================

def image_negative(img):
    """
    Compute the photographic negative: s = 255 - r
    Works on both grayscale and RGB (per-channel).
    
    Reference: Gonzalez & Woods (2018), Section 3.2.1
    """
    return 255 - img

# --- Grayscale Demo: Moon ---
neg_moon = image_negative(img_moon)
show_with_histogram(img_moon, neg_moon,
    'Image Negative - Moon (Grayscale)',
    processed_label='Negative',
    subtitle='s = 255 - r  |  Note the histogram is mirrored!')

# --- RGB Demo: Astronaut ---
neg_astro = image_negative(img_astronaut)
show_comparison(img_astronaut, neg_astro,
    'Image Negative - Astronaut (RGB)',
    processed_label='Negative',
    subtitle='Per-channel inversion: s_c = 255 - r_c for c in {R, G, B}')

# --- RGB Demo: Chelsea (cat) ---
neg_chelsea = image_negative(img_chelsea)
show_comparison(img_chelsea, neg_chelsea,
    'Image Negative - Chelsea Cat (RGB)',
    processed_label='Negative',
    subtitle='Color inversion creates an eerie complementary-color effect')
# --- RGB Demo: Jebena ---
neg_jebena = image_negative(img_jebena)
show_comparison(img_jebena, neg_jebena,
    'Image Negative - Jebena (RGB)',
    processed_label='Negative')


# 2. ️ Gamma Encoding / Correction

## Theory

The **power-law (gamma) transformation** is defined as:

$$s = c \cdot r^\gamma$$

where $r$ is normalized to $[0, 1]$ and the result is rescaled to $[0, 255]$.

| $\gamma$ | Effect |
|----------|--------|
| $\gamma < 1$ | Brightens dark regions (expands low intensities) |
| $\gamma = 1$ | Identity transformation |
| $\gamma > 1$ | Darkens the image (compresses low intensities) |

### Historical Context
- **CRT monitors** exhibit inherent gamma ≈ 2.2 (nonlinear voltage-to-brightness)
- The **sRGB standard** (IEC 61966-2-1) encodes gamma correction into the color space
- **Poynton (1998)** provides the definitive treatment of gamma in imaging systems

> *References: Gonzalez & Woods (2018) §3.2.3; Poynton, C. (1998). "The Rehabilitation of Gamma"*

In [ ]:
# ============================================================
# 2. GAMMA ENCODING / CORRECTION
# s = c * r^gamma (power-law transformation)
# ============================================================

def gamma_correction(img, gamma=1.0, c=1.0):
    """
    Apply power-law (gamma) transformation.
    s = c * r^gamma, where r is normalized to [0, 1].
    
    References:
      - Gonzalez & Woods (2018), Section 3.2.3
      - Poynton, C. (1998). "The Rehabilitation of Gamma."
      - IEC 61966-2-1:1999 (sRGB standard)
    """
    normalized = img.astype(np.float64) / 255.0
    corrected = c * np.power(normalized, gamma)
    return (corrected * 255).clip(0, 255).astype(np.uint8)

# --- Grayscale Demo: Cameraman ---
gammas = [0.1, 0.3, 0.5, 1.0, 1.5, 2.0, 3.0]
images = [img_camera] + [gamma_correction(img_camera, g) for g in gammas]
labels = ['Original'] + [f'\u03b3 = {g}' for g in gammas]
show_multi(images, labels,
    'Gamma Correction - Cameraman (Grayscale)',
    cols=4,
    subtitle='Power-law transform: \u03b3 < 1 brightens, \u03b3 > 1 darkens  |  Poynton (1998)')

# --- RGB Demo: Coffee ---
images_c = [img_coffee] + [gamma_correction(img_coffee, g) for g in gammas]
show_multi(images_c, labels,
    'Gamma Correction - Coffee (RGB)',
    cols=4,
    subtitle='sRGB standard uses \u03b3 \u2248 2.2 for CRT compensation  |  IEC 61966-2-1')

# --- Histogram comparison for key gamma values ---
show_with_histogram(img_camera, gamma_correction(img_camera, 0.3),
    'Gamma = 0.3 (Brightening) - Cameraman',
    processed_label='\u03b3 = 0.3',
    subtitle='Low gamma expands dark intensities -> brighter image')

show_with_histogram(img_camera, gamma_correction(img_camera, 2.5),
    'Gamma = 2.5 (Darkening) - Cameraman',
    processed_label='\u03b3 = 2.5',
    subtitle='High gamma compresses dark intensities -> darker image')
# --- RGB Demo: Jebena ---
images_j = [img_jebena] + [gamma_correction(img_jebena, g) for g in gammas]
show_multi(images_j, labels,
    'Gamma Correction - Jebena (RGB)',
    cols=4)


# 3.  Logarithmic Transformation

## Theory

The **logarithmic transformation** compresses the dynamic range:

$$s = c \cdot \log_2(1 + r)$$

where $c = \frac{255}{\log_2(1 + \max(r))}$ for full dynamic range utilization.

### Scientific Foundation

- **Stockham (1972)** demonstrated that logarithmic processing models human visual perception
- **Weber-Fechner Law**: perceived brightness $\propto \log(\text{actual intensity})$
- Essential for displaying **Fourier magnitude spectra** (values span $10^6$ range)
- The **inverse log** (exponential) produces the opposite effect

> *Reference: Stockham, T.G. Jr. (1972). "Image Processing in the Context of a Visual Model." Proc. IEEE, 60(7), 828-842.*

In [ ]:
# ============================================================
# 3. LOGARITHMIC TRANSFORMATION
# s = c * log2(1 + r)
# ============================================================

def log_transform(img, c=None):
    """
    Apply logarithmic intensity transformation.
    s = c * log2(1 + r)
    
    Reference: Stockham (1972); Gonzalez & Woods (2018), Section 3.2.2
    """
    img_float = img.astype(np.float64)
    if c is None:
        max_val = np.max(img_float)
        c = 255.0 / np.log2(1 + max_val) if max_val > 0 else 1.0
    result = c * np.log2(1 + img_float)
    return result.clip(0, 255).astype(np.uint8)

def inverse_log_transform(img):
    """
    Apply inverse logarithmic (exponential) transformation.
    Expands higher intensity values, compresses lower ones.
    """
    img_float = img.astype(np.float64) / 255.0
    result = np.power(2, img_float * 8) - 1
    if result.max() > 0:
        result = (result / result.max()) * 255
    return result.clip(0, 255).astype(np.uint8)

# --- Grayscale Demo: Moon ---
log_moon = log_transform(img_moon)
inv_log_moon = inverse_log_transform(img_moon)

show_with_histogram(img_moon, log_moon,
    'Log Transform - Moon (Grayscale)',
    processed_label='Log Transformed',
    subtitle='s = c * log2(1 + r)  |  Stockham (1972) -- Weber-Fechner Law')

show_multi(
    [img_moon, log_moon, inv_log_moon],
    ['Original', 'Log Transform', 'Inverse Log'],
    'Log vs. Inverse Log - Moon',
    cols=3,
    subtitle='Log compresses highlights; Inverse log expands them')

# --- RGB Demo: Astronaut ---
log_astro = log_transform(img_astronaut)
show_with_histogram(img_astronaut, log_astro,
    'Log Transform - Astronaut (RGB)',
    processed_label='Log Transformed',
    subtitle='Dynamic range compression applied per-channel')
# --- RGB Demo: Jebena ---
log_jebena = log_transform(img_jebena)
show_with_histogram(img_jebena, log_jebena,
    'Log Transform - Jebena (RGB)',
    processed_label='Log Transformed')


# 4.  Contrast Stretching

## Theory

**Contrast stretching** expands a narrow intensity range to span $[0, 255]$ using a **piecewise linear** mapping:

$$s = 255 \cdot \frac{r - r_{\min}}{r_{\max} - r_{\min}}$$

### Three Variants
1. **Min-Max**: Uses absolute min/max — sensitive to outliers
2. **Percentile (2%-98%)**: Uses percentile boundaries — robust to outliers
3. **Piecewise Linear**: Custom breakpoints $(r_1, s_1)$, $(r_2, s_2)$ for fine control

### When to Use
- Histogram equalization produces over-enhanced or unnatural results
- You need predictable, linear contrast adjustment
- Washed-out or low-contrast images (satellite, medical)

> *Reference: Gonzalez & Woods (2018), §3.2.4*

In [ ]:
# ============================================================
# 4. CONTRAST STRETCHING
# Piecewise linear mapping
# ============================================================

def contrast_stretch_minmax(img):
    """Min-Max contrast stretching: s = 255 * (r - r_min) / (r_max - r_min)"""
    def _stretch(ch):
        r_min, r_max = ch.min(), ch.max()
        if r_max == r_min: return ch
        return ((ch.astype(np.float64) - r_min) / (r_max - r_min) * 255).clip(0, 255).astype(np.uint8)
    if len(img.shape) == 2:
        return _stretch(img)
    return np.stack([_stretch(img[:,:,i]) for i in range(img.shape[2])], axis=2)

def contrast_stretch_percentile(img, low_pct=2, high_pct=98):
    """Percentile-based contrast stretching (robust to outliers)."""
    def _stretch(ch):
        p_low = np.percentile(ch, low_pct)
        p_high = np.percentile(ch, high_pct)
        if p_high == p_low: return ch
        return ((ch.astype(np.float64) - p_low) / (p_high - p_low) * 255).clip(0, 255).astype(np.uint8)
    if len(img.shape) == 2:
        return _stretch(img)
    return np.stack([_stretch(img[:,:,i]) for i in range(img.shape[2])], axis=2)

def contrast_stretch_piecewise(img, r1=70, s1=0, r2=180, s2=255):
    """Piecewise linear contrast stretching with custom breakpoints."""
    lut = np.zeros(256, dtype=np.uint8)
    for r in range(256):
        if r < r1:
            lut[r] = int(s1 / max(r1, 1) * r) if r1 > 0 else 0
        elif r <= r2:
            lut[r] = int(s1 + (s2 - s1) / max(r2 - r1, 1) * (r - r1))
        else:
            lut[r] = int(s2 + (255 - s2) / max(255 - r2, 1) * (r - r2))
    if len(img.shape) == 2:
        return cv2.LUT(img, lut)
    return np.stack([cv2.LUT(img[:,:,i], lut) for i in range(img.shape[2])], axis=2)

# --- Grayscale Demo: Moon (low contrast) ---
stretched_mm = contrast_stretch_minmax(img_moon)
stretched_pct = contrast_stretch_percentile(img_moon, 2, 98)
stretched_pw = contrast_stretch_piecewise(img_moon, r1=80, s1=10, r2=180, s2=245)

show_multi(
    [img_moon, stretched_mm, stretched_pct, stretched_pw],
    ['Original', 'Min-Max', 'Percentile (2%-98%)', 'Piecewise Linear'],
    'Contrast Stretching Comparison - Moon',
    cols=4,
    subtitle='Three variants: Min-Max | Percentile (robust) | Piecewise Linear (custom)')

show_with_histogram(img_moon, stretched_pct,
    'Contrast Stretching - Moon (Percentile)',
    processed_label='Percentile Stretched',
    subtitle='2nd-98th percentile stretching - robust to outliers')

# --- RGB Demo: Astronaut ---
stretched_astro = contrast_stretch_percentile(img_astronaut)
show_with_histogram(img_astronaut, stretched_astro,
    'Contrast Stretching - Astronaut (RGB)',
    processed_label='Percentile Stretched',
    subtitle='Per-channel percentile stretching preserves color balance')
# --- RGB Demo: Jebena ---
stretched_jebena = contrast_stretch_piecewise(img_jebena, r1=50, s1=10, r2=200, s2=245)
show_with_histogram(img_jebena, stretched_jebena,
    'Contrast Stretching - Jebena (Piecewise)',
    processed_label='Piecewise Stretched')


# 5.  Histogram Equalization

## Theory

**Histogram equalization** automatically adjusts contrast by mapping the intensity distribution to be approximately **uniform**. The transformation function is the **CDF** of the input histogram:

$$s_k = (L-1) \cdot \sum_{j=0}^{k} p(r_j) \quad \text{where } p(r_j) = \frac{h(r_j)}{M \times N}$$

### Algorithm
1. Compute histogram $h(r_k)$ for each intensity level
2. Normalize to PDF: $p(r_j) = h(r_j) / (M \times N)$
3. Compute CDF: $s_k = \sum_{j=0}^{k} p(r_j)$
4. Map: $\text{output} = \text{round}((L-1) \times s_k)$

### Global HE vs. CLAHE
- **Global HE**: One transformation for entire image — can over-enhance
- **CLAHE** (Pizer et al., 1987): Divides image into tiles, equalizes locally with contrast clip limit
- **For RGB**: Convert to HSV, equalize only V channel to preserve hue & saturation

> *Reference: Pizer, S.M. et al. (1987). "Adaptive Histogram Equalization and Its Variations." CVGIP, 39, 355-368.*

In [ ]:
# ============================================================
# 5. HISTOGRAM EQUALIZATION
# CDF-based mapping for uniform intensity distribution
# ============================================================

def histogram_equalization_manual(img):
    """
    Manual histogram equalization using CDF computation.
    For RGB: equalize V channel in HSV space.
    
    Reference: Gonzalez & Woods (2018), Section 3.3.1
    """
    def _equalize(channel):
        hist, bins = np.histogram(channel.flatten(), 256, [0, 256])
        cdf = hist.cumsum()
        cdf_masked = np.ma.masked_equal(cdf, 0)
        cdf_norm = ((cdf_masked - cdf_masked.min()) * 255 /
                    (cdf_masked.max() - cdf_masked.min()))
        cdf_final = np.ma.filled(cdf_norm, 0).astype(np.uint8)
        return cdf_final[channel]
    
    if len(img.shape) == 2:
        return _equalize(img)
    else:
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        hsv[:,:,2] = _equalize(hsv[:,:,2])
        return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

def clahe_transform(img, clip_limit=2.0, tile_grid_size=(8,8)):
    """
    CLAHE - Contrast Limited Adaptive Histogram Equalization.
    
    Reference: Pizer et al. (1987)
    """
    clahe_obj = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    if len(img.shape) == 2:
        return clahe_obj.apply(img)
    else:
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        hsv[:,:,2] = clahe_obj.apply(hsv[:,:,2])
        return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

# --- Grayscale Demo: Moon ---
he_manual = histogram_equalization_manual(img_moon)
he_opencv = cv2.equalizeHist(img_moon)
he_clahe = clahe_transform(img_moon)

show_multi(
    [img_moon, he_manual, he_opencv, he_clahe],
    ['Original', 'Manual HE', 'OpenCV HE', 'CLAHE'],
    'Histogram Equalization - Moon (Grayscale)',
    cols=4,
    subtitle='Global HE vs. CLAHE (Pizer et al., 1987)  |  CDF -> Uniform distribution')

show_with_histogram(img_moon, he_manual,
    'Histogram Equalization - Moon (Manual CDF)',
    processed_label='Manual HE',
    subtitle='sk = Sum p(rj) mapped to [0, 255]  |  CDF-based equalization')

show_with_histogram(img_moon, he_clahe,
    'CLAHE - Moon',
    processed_label='CLAHE',
    subtitle='Contrast Limited AHE - clip=2.0, tile=8x8  |  Pizer et al. (1987)')

# --- RGB Demo: Astronaut ---
he_astro = histogram_equalization_manual(img_astronaut)
clahe_astro = clahe_transform(img_astronaut)

show_multi(
    [img_astronaut, he_astro, clahe_astro],
    ['Original', 'Global HE (HSV-V)', 'CLAHE (HSV-V)'],
    'Histogram Equalization - Astronaut (RGB)',
    cols=3,
    subtitle='For RGB: equalize V channel in HSV space to preserve hue & saturation')
# --- RGB Demo: Jebena ---
he_jebena = histogram_equalization_manual(img_jebena)
clahe_jebena = clahe_transform(img_jebena)
show_multi(
    [img_jebena, he_jebena, clahe_jebena],
    ['Original', 'Global HE', 'CLAHE'],
    'Histogram Equalization - Jebena (RGB)', cols=3)


# 6.  Intensity Level Slicing

## Theory

**Intensity level slicing** highlights a specific range of gray levels:

**Variant A** — Without Background:
$$s = \begin{cases} \text{highlight} & \text{if } \text{low} \leq r \leq \text{high} \\ 0 & \text{otherwise} \end{cases}$$

**Variant B** — With Background Preservation:
$$s = \begin{cases} \text{highlight} & \text{if } \text{low} \leq r \leq \text{high} \\ r & \text{otherwise} \end{cases}$$

### Applications
- **Medical imaging**: Isolate tissue types (bone vs. soft tissue)
- **Quality inspection**: Highlight defects
- **Geospatial**: Classify terrain in satellite imagery

> *Reference: Gonzalez & Woods (2018), §3.2.5*

In [ ]:
# ============================================================
# 6. INTENSITY LEVEL SLICING
# Highlight a specific range of intensities
# ============================================================

def intensity_slice_binary(img, low=100, high=200, highlight_val=255):
    """Binary slicing: pixels in [low,high] -> highlight, others -> 0."""
    gray = img if len(img.shape) == 2 else cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    result = np.zeros_like(gray)
    result[(gray >= low) & (gray <= high)] = highlight_val
    return result

def intensity_slice_preserve(img, low=100, high=200, highlight_val=255):
    """Slicing with background preservation."""
    gray = img if len(img.shape) == 2 else cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    result = gray.copy()
    result[(gray >= low) & (gray <= high)] = highlight_val
    return result

# --- Grayscale Demo: Camera with multiple ranges ---
for low, high in [(50, 120), (100, 200), (150, 220)]:
    binary = intensity_slice_binary(img_camera, low, high)
    preserve = intensity_slice_preserve(img_camera, low, high)
    show_multi(
        [img_camera, binary, preserve],
        ['Original', f'Binary [{low}-{high}]', f'Preserved [{low}-{high}]'],
        f'Intensity Level Slicing - Camera [{low}-{high}]',
        cols=3,
        subtitle=f'Range [{low}, {high}]: Binary (no bg) vs. Preserved (with bg)')

# --- RGB Demo: Astronaut ---
binary_a = intensity_slice_binary(img_astronaut, 100, 200)
preserve_a = intensity_slice_preserve(img_astronaut, 100, 200)
show_multi(
    [img_astronaut, binary_a, preserve_a],
    ['Original (RGB)', 'Binary [100-200]', 'Preserved [100-200]'],
    'Intensity Level Slicing - Astronaut',
    cols=3,
    subtitle='RGB converted to grayscale for slicing  |  Medical imaging application')
# --- RGB Demo: Jebena ---
binary_j = intensity_slice_binary(img_jebena, 100, 200)
preserve_j = intensity_slice_preserve(img_jebena, 100, 200)
show_multi(
    [img_jebena, binary_j, preserve_j],
    ['Original', 'Binary [100-200]', 'Preserved [100-200]'],
    'Intensity Level Slicing - Jebena', cols=3)


# 7.  Bit Plane Slicing

## Theory

An 8-bit grayscale image can be decomposed into **8 binary images** (bit planes):

$$\text{pixel} = a_7 \cdot 2^7 + a_6 \cdot 2^6 + \ldots + a_1 \cdot 2^1 + a_0 \cdot 2^0$$

| Bit Plane | Significance | Visual Quality |
|-----------|-------------|----------------|
| Bit 7 (MSB) | 50% of total intensity | Coarse structure |
| Bits 4-6 | Significant detail | Recognizable features |
| Bits 0-3 (LSB) | Fine texture/noise | Often discardable |

### Applications
- **Image compression**: Discard lower bit planes (lossy but acceptable)
- **Steganography/Watermarking**: Embed data in LSB planes (invisible to eye)
- **Analysis**: Understand contribution of each bit to visual quality

> *Reference: Gonzalez & Woods (2018), §3.2.6*

In [ ]:
# ============================================================
# 7. BIT PLANE SLICING
# Decompose image into individual bit planes
# ============================================================

def bit_plane_slice(img, bit):
    """Extract a single bit plane (0=LSB, 7=MSB)."""
    gray = img if len(img.shape) == 2 else cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    plane = (gray >> bit) & 1
    return (plane * 255).astype(np.uint8)

def bit_plane_decompose(img):
    """Decompose into all 8 bit planes (LSB to MSB)."""
    return [bit_plane_slice(img, bit) for bit in range(8)]

def bit_plane_reconstruct(img, num_planes=4):
    """Reconstruct from top-k MSB planes."""
    gray = img if len(img.shape) == 2 else cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    result = np.zeros_like(gray, dtype=np.uint8)
    for bit in range(8 - num_planes, 8):
        plane = (gray >> bit) & 1
        result += (plane << bit).astype(np.uint8)
    return result

# --- Grayscale Demo: Cameraman ---
planes = bit_plane_decompose(img_camera)
images_bp = [img_camera] + planes[::-1]  # MSB first
labels_bp = ['Original'] + [f'Bit {7-i}{" (MSB)" if i==0 else " (LSB)" if i==7 else ""}'
             for i in range(8)]

show_multi(images_bp, labels_bp,
    'Bit Plane Decomposition - Cameraman',
    cols=3,
    subtitle='pixel = a7*2^7 + a6*2^6 + ... + a1*2^1 + a0*2^0  |  MSB carries most info')

In [ ]:
# --- Reconstruction from top-k planes ---
reconstructions = [bit_plane_reconstruct(img_camera, k) for k in [1, 2, 3, 4, 5, 6, 7, 8]]
images_r = [img_camera] + reconstructions
labels_r = ['Original'] + [f'Top {k} planes' for k in [1, 2, 3, 4, 5, 6, 7, 8]]

show_multi(images_r, labels_r,
    'Bit Plane Reconstruction - Cameraman',
    cols=3,
    subtitle='Progressive reconstruction: top-4 planes ~ recognizable  |  Compression insight')

In [ ]:
# --- RGB Demo: Astronaut per-channel MSB ---
r_msb = bit_plane_slice(img_astronaut[:,:,0], 7)
g_msb = bit_plane_slice(img_astronaut[:,:,1], 7)
b_msb = bit_plane_slice(img_astronaut[:,:,2], 7)
gray_msb = bit_plane_slice(cv2.cvtColor(img_astronaut, cv2.COLOR_RGB2GRAY), 7)

show_multi(
    [img_astronaut, r_msb, g_msb, b_msb, gray_msb],
    ['Original RGB', 'R Channel MSB', 'G Channel MSB', 'B Channel MSB', 'Gray MSB'],
    'Bit Plane Slicing (MSB) - Astronaut Per-Channel',
    cols=3,
    subtitle='MSB (Bit 7) per RGB channel vs. grayscale  |  Channel-wise decomposition')
# --- RGB Demo: Jebena per-channel MSB ---
r_msb_j = bit_plane_slice(img_jebena[:,:,0], 7)
g_msb_j = bit_plane_slice(img_jebena[:,:,1], 7)
b_msb_j = bit_plane_slice(img_jebena[:,:,2], 7)
gray_msb_j = bit_plane_slice(cv2.cvtColor(img_jebena, cv2.COLOR_RGB2GRAY), 7)
show_multi(
    [img_jebena, r_msb_j, g_msb_j, b_msb_j, gray_msb_j],
    ['Original', 'R Channel MSB', 'G Channel MSB', 'B Channel MSB', 'Gray MSB'],
    'Bit Plane Slicing (MSB) - Jebena', cols=3)


#  Summary & Comparative Analysis

## Method Comparison on Moon Image (Low Contrast)

Let's apply all methods to the same image for a side-by-side comparison.

In [ ]:
# Apply all methods to the Moon image for comparison
results = [
    img_moon,
    image_negative(img_moon),
    gamma_correction(img_moon, 0.4),
    log_transform(img_moon),
    contrast_stretch_percentile(img_moon),
    histogram_equalization_manual(img_moon),
    clahe_transform(img_moon),
    intensity_slice_preserve(img_moon, 100, 200),
    bit_plane_slice(img_moon, 7),
]
result_labels = [
    'Original', 'Negative', 'Gamma (0.4)',
    'Log Transform', 'Contrast Stretch', 'Hist. Equalization',
    'CLAHE', 'Intensity Slice', 'MSB Bit Plane'
]

show_multi(results, result_labels,
    'Comparative Analysis: All Methods on Moon Image',
    cols=3,
    subtitle='Each method reveals different aspects of the underlying image data')

#  Fun Facts & Takeaways

### Key Insight
All 7 methods are **point operations** — they can be implemented as a single **256-entry Look-Up Table (LUT)**, making them **O(1) per pixel** after setup!

### Did You Know?
- The **Lena image** has been used in over **10,000 research papers** since 1973
- CRT monitors have an inherent gamma of ~2.2, which is why gamma correction exists
- **CLAHE** was originally developed for **medical imaging** (enhancing chest X-rays)
- The Weber-Fechner Law (1860) predates digital imaging by over a century!

